# E6 — MoE Routing Variance Ablation

**Tinker RL Project — PES University MTech Capstone (Group 6)**

| Field | Value |
|-------|-------|
| **Hypothesis** | High within-group reward variance in Qwen3-30B MoE training is caused by routing divergence |
| **Model** | `Qwen/Qwen3-30B-A3B` (MoE, 3B active params) |
| **Dataset** | GSM8K |
| **Ablation** | rollout temperature: 1.0 (standard) vs 0.3 (constrained routing) |
| **Key metric** | `train/reward_std` — within-group reward std per step |
| **Finding it tests** | Finding 4: MoE routing divergence compounds GRPO gradient variance |

**Run twice**: once with `ROLLOUT_TEMPERATURE=1.0`, once with `ROLLOUT_TEMPERATURE=0.3`.
Compare `train/reward_std` trajectories. Lower temp → more consistent routing → smoother curve.

In [ ]:
!pip install -q atroposlib==0.3.0 \
    'git+https://github.com/thinking-machines-lab/tinker.git' \
    datasets transformers wandb pydantic python-dotenv math-verify latex2sympy2-extended

In [ ]:
!git clone https://github.com/pes-llm-research/tinker-rl-lab.git
%cd tinker-rl-lab/atropos
!pip install -e . -q

In [ ]:
import os

os.environ['TINKER_API_KEY'] = 'YOUR_TINKER_API_KEY'
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN'

# *** CHANGE to '0.3' for the second run ***
TEMPERATURE = '1.0'   # or '0.3'
os.environ['ROLLOUT_TEMPERATURE'] = TEMPERATURE

config_suffix = TEMPERATURE.replace('.', '_')
config = f'configs/moe_routing_temp{config_suffix}.yaml'
os.environ['TINKER_CONFIG_PATH'] = config
print(f'Rollout temperature: {TEMPERATURE} | config: {config}')

In [ ]:
!nohup python -m atroposlib.server > /tmp/atropos_server.log 2>&1 &
import time; time.sleep(5)
print('Atropos coordinator started')

In [ ]:
import os
config = os.environ['TINKER_CONFIG_PATH']
!nohup python -m tinker_atropos.environments.moe_routing_tinker \
    --config {config} \
    > /tmp/moe_routing_env.log 2>&1 &
import time; time.sleep(10)
print('Environment started')
!tail -5 /tmp/moe_routing_env.log

In [ ]:
import os
config = os.environ['TINKER_CONFIG_PATH']
!python launch_training.py --config {config}

In [ ]:
# RESULTS — paste wandb data for both temperature runs

results = {
    'temp_1_0': {
        'steps': [],
        'rewards': [],
        'reward_std': [],    # from wandb train/reward_std
        'frac_volatile': [], # from wandb train/frac_volatile
    },
    'temp_0_3': {
        'steps': [],
        'rewards': [],
        'reward_std': [],
        'frac_volatile': [],
    },
}

if results['temp_1_0']['steps'] and results['temp_0_3']['steps']:
    import matplotlib.pyplot as plt
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(results['temp_1_0']['steps'], results['temp_1_0']['rewards'], color='#e74c3c', linewidth=2, label='temp=1.0 (standard)')
    ax1.plot(results['temp_0_3']['steps'], results['temp_0_3']['rewards'], color='#3498db', linewidth=2, label='temp=0.3 (constrained)')
    ax1.set_xlabel('Step'); ax1.set_ylabel('Accuracy'); ax1.legend(); ax1.grid(True, alpha=0.3)
    ax1.set_title('Accuracy Trajectory')

    ax2.plot(results['temp_1_0']['steps'], results['temp_1_0']['reward_std'], color='#e74c3c', linewidth=2, label='temp=1.0')
    ax2.plot(results['temp_0_3']['steps'], results['temp_0_3']['reward_std'], color='#3498db', linewidth=2, label='temp=0.3')
    ax2.set_xlabel('Step'); ax2.set_ylabel('Within-Group Reward Std'); ax2.legend(); ax2.grid(True, alpha=0.3)
    ax2.set_title('Within-Group Reward Variance (MoE Routing Signal)')

    plt.suptitle('E6: MoE Routing Variance — Qwen3-30B on GSM8K', fontsize=13)
    plt.tight_layout()
    plt.show()

    for t, key in [('1.0', 'temp_1_0'), ('0.3', 'temp_0_3')]:
        r = results[key]
        if r['reward_std']:
            print(f'temp={t}: avg reward_std={sum(r["reward_std"])/len(r["reward_std"]):.3f}')
else:
    print('Run both temperature conditions first.')